# 05 - Interpretabilidad, Explicabilidad y Valor de Negocio

En este último notebook del pipeline de Machine Learning, el objetivo principal es dotar de transparencia a nuestras predicciones algorítmicas. Esto resulta ser un paso fundamental crítico para la adopción del modelo por parte de los stakeholders de negocio, como los Gerentes de Inventario y Supply Chain.

Abordaremos dos componentes clave:
1. **Feature Importance Global:** Analizar qué atributos, factores demográficos y rezagos temporales (lags) aportan la mayor ganancia de información (Gain) sistemática en la reducción de nuestro WMAE.
2. **Validación Visual de Series (Business Eye-Test):** Comparar analíticamente el histórico real de ventas frente a la trayectoria proyectada a 39 semanas para combinaciones específicas de Tienda-Departamento, comprobando la robustez del forecast en los picos estacionales álgidos (Thanksgiving, Christmas).

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')

## 1. Carga de Artefactos de Modelo (Fase de Inferencia)
Aplicamos los principios fundamentales de MLOps al separar la fase computacional de entrenamiento (Training) de la ejecución de inferencias (Scoring) gracias a la carga de modelos previamente serializados. Esto escala mejor en la infraestructura de producción.

In [ ]:
MODELS_DIR = "../models"

# Cargar LightGBM
lgb_model = lgb.Booster(model_file=f"{MODELS_DIR}/lightgbm_final.txt")
print("LightGBM cargado desde disco.")

# Cargar XGBoost
xgb_model = xgb.Booster(model_file=f"{MODELS_DIR}/xgboost_final.json")
print("XGBoost cargado.")

# Leer los pesos (0.60 para LGBM)
weight_lgb = 0.60
weight_xgb = 0.40

## 2. Análisis de Importancia de Variables (Feature Importance - Gain)
Estimamos el valor explicativo derivado mediante la métrica de **Ganancia de Información (Gain)**. A diferencia del cálculo simple por frecuencia (Weight - veces que se usa un feature para ramificar), la *Ganancia* representa de manera intrínseca cuánto contribuyó marginalmente esa métrica a minimizar el error absoluto de la función de coste.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

# Plot LightGBM Importance (Gain: Cuánto ayudó a reducir el error la variable)
lgb.plot_importance(lgb_model, importance_type='gain', max_num_features=15, 
                    title="LightGBM - Importancia (Ganancia)", ax=ax[0], color='cornflowerblue')

# Plot XGBoost Importance
xgb.plot_importance(xgb_model, importance_type='gain', max_num_features=15, 
                    title="XGBoost - Importancia (Ganancia)", ax=ax[1], color='coral')

plt.tight_layout()
plt.show()

## 3. Validación Visual de Pronósticos (Business Eye-Test)
Más allá de las métricas deterministas agregadas como el WMAE, un catalizador esencial para la adopción del producto de datos recae en el juicio analítico visual. 

En esta sección unificamos el ecosistema del registro histórico de entrenamiento y las trayectorias de testeo predichas. Este análisis nos permite evaluar visualmente si el modelo logra captar las inflexiones estacionales y evitar predicciones anómalas en distintos tipos de departamentos.

In [ ]:
# 1. Cargar el histórico real (Train) y convertir a formato útil
train = pd.read_csv("../data/raw/train.csv")
train['Date'] = pd.to_datetime(train['Date'])

# 2. Cargar nuestra submission final
sub = pd.read_csv("../outputs/submission.csv")

# Desempacar la columna 'Id' (Store_Dept_Date)
# Usamos expansión por guion bajo
sub_info = sub['Id'].str.split('_', expand=True)
sub['Store'] = sub_info[0].astype(int)
sub['Dept'] = sub_info[1].astype(int)
sub['Date'] = pd.to_datetime(sub_info[2])

print("Datos cargados para visualización.")

In [ ]:
def plot_store_dept_forecast(store_id, dept_id):
    # Extraer historia
    hist = train[(train['Store'] == store_id) & (train['Dept'] == dept_id)].sort_values('Date')
    
    # Extraer pronóstico
    forecast = sub[(sub['Store'] == store_id) & (sub['Dept'] == dept_id)].sort_values('Date')
    
    if len(hist) == 0 or len(forecast) == 0:
        print(f"[{store_id}-{dept_id}] Datos insuficientes para renderizar la comparativa.")
        return
    
    plt.figure(figsize=(14, 5))
    # Graficar historia
    plt.plot(hist['Date'], hist['Weekly_Sales'], label='Ventas Históricas (Real)', color='black', marker='.')
    
    # Graficar pronóstico
    plt.plot(forecast['Date'], forecast['Weekly_Sales'], label='Forecast (Ensemble)', color='green', linestyle='--', marker='o')
    
    # Delimitador temporal (horizonte de corte Train/Test)
    split_date = hist['Date'].max()
    plt.axvline(split_date, color='red', linestyle=':', label='Fin de Datos Históricos')
    
    plt.title(f"Pronóstico de Ventas Semanales | Tienda: {store_id} - Departamento: {dept_id}", fontsize=14)
    plt.xlabel("Fecha")
    plt.ylabel("Ventas en $USD")
    plt.legend()
    plt.show()

# Evaluación visual sobre un departamento con comportamiento estacional estándar
plot_store_dept_forecast(1, 1)

In [ ]:
# Evaluación visual sobre un departamento con comportamiento estacional extremo o volátil (ej. Holidays)
plot_store_dept_forecast(20, 92)